# Wan 2.1 Video Generation API
**BlueRidgeCustomCo — TikTok Product Video Generator**

This notebook runs Wan 2.1 image-to-video on Colab's free T4 GPU and exposes a Gradio API endpoint your VPS can call.

## Setup
1. Go to **Runtime → Change runtime type → T4 GPU** (or A100 if available)
2. Run all cells in order
3. Copy the Gradio public URL from the last cell output
4. Paste it into your vinylApp `.env` as `WAN_VIDEO_API_URL`

In [ ]:
#@title 1. Install Dependencies
!pip install -q gradio diffusers transformers accelerate pillow imageio
!pip install -q huggingface_hub sentencepiece
print('\n✅ Dependencies installed')

In [ ]:
#@title 2. Load Wan2.1 Image-to-Video Pipeline
import torch
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
from diffusers.utils import load_image
from transformers import CLIPVisionModel
import gc

# The -Diffusers suffix is required for HuggingFace diffusers-compatible weights
MODEL_ID = "Wan-AI/Wan2.1-I2V-14B-480P-Diffusers"

print(f"Loading {MODEL_ID}...")
print("This will download ~28GB on first run (cached after).")

# Load image encoder and VAE with float32 (required for these components)
image_encoder = CLIPVisionModel.from_pretrained(
    MODEL_ID, subfolder="image_encoder", torch_dtype=torch.float32
)
vae = AutoencoderKLWan.from_pretrained(
    MODEL_ID, subfolder="vae", torch_dtype=torch.float32
)

# Load full pipeline with bfloat16 for the transformer
pipe = WanImageToVideoPipeline.from_pretrained(
    MODEL_ID,
    vae=vae,
    image_encoder=image_encoder,
    torch_dtype=torch.bfloat16
)

# Enable memory optimizations for free T4 (16GB)
pipe.enable_model_cpu_offload()

gc.collect()
torch.cuda.empty_cache()

# Check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
print(f"\n✅ Pipeline loaded: {MODEL_ID}")

In [ ]:
#@title 3. Launch Gradio Video Generation API
import gradio as gr
import numpy as np
import time
import uuid
import json
import os
import imageio
from PIL import Image
from diffusers.utils import export_to_video

OUTPUT_DIR = "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Optimal sizes for Wan2.1 I2V 480P
RESOLUTIONS = {
    "832x480 (landscape)": (832, 480),
    "480x832 (portrait/TikTok)": (480, 832),
    "624x624 (square)": (624, 624)
}

def generate_video(
    image,
    prompt="The product gently rotates with subtle movement and a soft zoom effect.",
    resolution="480x832 (portrait/TikTok)",
    num_frames=41,
    guidance_scale=5.0
):
    """Generate a video from an input image using Wan2.1"""
    start = time.time()
    job_id = str(uuid.uuid4())[:8]

    try:
        # Prepare input image
        if isinstance(image, np.ndarray):
            image = Image.fromarray(image)

        w, h = RESOLUTIONS.get(resolution, (480, 832))
        image = image.resize((w, h), Image.LANCZOS)

        print(f"[{job_id}] Generating video...")
        print(f"  Prompt: {prompt}")
        print(f"  Resolution: {w}x{h}, Frames: {num_frames}")

        # Generate
        output = pipe(
            image=image,
            prompt=prompt,
            negative_prompt="Bright tones, overexposed, static, blurry, bad quality, distorted",
            height=h,
            width=w,
            num_frames=num_frames,
            guidance_scale=guidance_scale,
            num_inference_steps=20
        ).frames[0]

        # Save video
        output_path = f"{OUTPUT_DIR}/{job_id}.mp4"
        export_to_video(output, output_path, fps=16)

        elapsed = time.time() - start
        print(f"[{job_id}] ✅ Done in {elapsed:.0f}s")

        gc.collect()
        torch.cuda.empty_cache()

        return output_path

    except Exception as e:
        print(f"[{job_id}] ❌ Error: {e}")
        gc.collect()
        torch.cuda.empty_cache()
        raise gr.Error(f"Generation failed: {e}")


def health_check():
    import subprocess
    return json.dumps({
        "status": "ok",
        "model": "Wan2.1-I2V-14B-480P",
        "gpu": subprocess.getoutput("nvidia-smi --query-gpu=name,memory.free --format=csv,noheader")
    })


# Build Gradio interface
with gr.Blocks(title="Wan2.1 Video API — BlueRidgeCustomCo") as app:
    gr.Markdown("## Wan 2.1 — Image to Video API")
    gr.Markdown("Upload a product image and describe the animation you want.")

    with gr.Row():
        with gr.Column():
            img_input = gr.Image(label="Input Image", type="numpy")
            prompt_input = gr.Textbox(
                label="Motion Prompt",
                value="The sticker design gently animates with a subtle 3D rotation, light reflections move across the surface revealing the glossy vinyl texture.",
                lines=3
            )
            resolution_input = gr.Dropdown(
                choices=list(RESOLUTIONS.keys()),
                value="480x832 (portrait/TikTok)",
                label="Resolution"
            )
            frames_input = gr.Slider(
                minimum=17, maximum=81, step=8, value=41,
                label="Frames (41≈2.5sec, 81≈5sec)"
            )
            guidance_input = gr.Slider(
                minimum=1.0, maximum=10.0, step=0.5, value=5.0,
                label="Guidance Scale (higher=more faithful to prompt)"
            )
            gen_btn = gr.Button("Generate Video", variant="primary")
        with gr.Column():
            video_output = gr.Video(label="Generated Video")

    gen_btn.click(
        fn=generate_video,
        inputs=[img_input, prompt_input, resolution_input, frames_input, guidance_input],
        outputs=video_output,
        api_name="generate"
    )

    health_btn = gr.Button("Health Check", visible=False)
    health_output = gr.Textbox(visible=False)
    health_btn.click(fn=health_check, outputs=health_output, api_name="health")

print("\n" + "="*60)
print("LAUNCHING GRADIO API")
print("Copy the public URL below into your .env as:")
print("WAN_VIDEO_API_URL=https://xxxxx.gradio.live")
print("="*60 + "\n")

app.launch(share=True, show_error=True)